# Assistive Health Event Narrative (AHEN)

Notebook dedicado a consulta das narrativas clinicas geradas na camada `data/enriched`.

In [ ]:
from pathlib import Path
import duckdb

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_GLOB = (PROJECT_ROOT / 'data' / 'processed' / '**' / '*.parquet').as_posix()
ENRICHED_GLOB = (PROJECT_ROOT / 'data' / 'enriched' / '**' / '*.parquet').as_posix()
ENRICHED_DIR = PROJECT_ROOT / 'data' / 'enriched'

def run_sql(sql: str):
    with duckdb.connect(database=':memory:') as con:
        con.execute(f"CREATE OR REPLACE VIEW processed AS SELECT * FROM read_parquet('{PROCESSED_GLOB}', union_by_name=true)")
        con.execute(f"CREATE OR REPLACE VIEW enriched AS SELECT * FROM read_parquet('{ENRICHED_GLOB}', union_by_name=true)")
        return con.execute(sql).df()

if not list(ENRICHED_DIR.rglob('*.parquet')):
    print(f'Nenhum parquet em {ENRICHED_DIR}. Execute: python -m src.data.clinical_inference')
else:
    print(f'Camada enriched encontrada em {ENRICHED_DIR}.')

In [ ]:
# Visao rapida: colunas e amostra\n
run_sql("""
SELECT *
FROM enriched
LIMIT 5
""")

In [ ]:
# Narrativas por competencia e tipo de atendimento
run_sql("""
SELECT
  clinical_tipo_atendimento,
  COUNT(*) AS n_registros,
  MIN(clinical_inference_version) AS versao
FROM enriched
GROUP BY clinical_tipo_atendimento
ORDER BY n_registros DESC
""")

In [ ]:
# Exemplos de transcricao para um recorte especifico
competencia = 202204
termo = 'osteomusculares'

run_sql(f"""
SELECT
  row_id,
  clinical_tipo_atendimento,
  clinical_interpretacao_clinica,
  clinical_event_narrative
FROM enriched
WHERE clinical_event_narrative ILIKE '%' || '{termo}' || '%'
LIMIT 10
""")

In [ ]:
# Join com processed para adicionar contexto tabular
competencia = 202204

run_sql(f"""
SELECT
  p.sistema,
  p.competencia_ano_mes,
  p.cid_principal,
  p.custo_total,
  e.clinical_tipo_atendimento,
  e.clinical_event_narrative
FROM processed p
JOIN enriched e USING (row_id)
WHERE p.competencia_ano_mes = {competencia}
LIMIT 20
""")